# Bước 2.3: tách mệnh đề

**Mục đích:** kiểm luật tách mệnh đề trước khi gán nhãn: mỗi bình luận ra bao nhiêu mệnh đề, mệnh đề dài bao nhiêu, cắt ở đâu, từ nối nào hay gặp, ví dụ, và chỗ có thể bỏ sót. Luật: `docs/2-tien-xu-ly/dac-ta-tach-menh-de.md`. Code: `src/processing/clauses.py`. Cập nhật lần cuối: 2026-11-19.

**Nội dung**

1. Đọc file đã xử lý
2. Số mệnh đề mỗi bình luận
3. Độ dài mệnh đề
4. Mệnh đề được cắt ở đâu
5. Từ nối (trên mẫu), và kiểm tra file khớp code
6. Ví dụ bình luận có nhiều mệnh đề
7. Chỗ có thể bỏ sót
8. Tổng hợp số liệu

**Đọc file nào:** `data/processed/comments.jsonl` và manifest của nó, do lệnh tầng 2 ghi ra. Chưa có file thì chạy trước, từ gốc repo: `.\.venv\Scripts\python.exe -m src.processing.run` (hướng dẫn tầng 2 mục 3). Notebook không ghi file nào.

In [ ]:
import json
import re
import unicodedata
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

from helpers import *   # ROOT, đường dẫn dữ liệu, N_SHOW, doc_jsonl, doc_da_xu_ly, doc_bo_may, lo_ra, bao_cao, display, pd
thiet_lap_hien_thi()

from src.processing import PIPELINE_VERSION, chuan_hoa, tach_emoji, tach_tu
from src.processing import clauses as L3   # 2.3 tách mệnh đề
from src.processing import clean as L1     # 2.1 làm sạch
from src.processing import normalize as L2   # 2.2 chuẩn hóa

### Tóm tắt luật tách

**Để làm gì:** mệnh đề là đơn vị gán nhãn khía cạnh và cảm xúc: "Cam đẹp nhưng pin yếu" → "Cam đẹp" | "nhưng pin yếu". Trước khi gán nhãn cần biết luật tách cho ra mệnh đề có hợp lý không: quá vụn, quá dài, hay cắt sai chỗ.

**Luật** (chi tiết ở `docs/2-tien-xu-ly/dac-ta-tach-menh-de.md`, code `src/processing/clauses.py`):
- cắt ở xuống dòng; dấu kết câu `. ! ? ; …` có khoảng trắng theo sau; dấu phẩy (trừ "12,5"); và trước từ nối tương phản (`nhưng`, `tuy nhiên`, `có điều`…);
- không cắt trong số, URL, email; không cắt ở "mà", "còn";
- mẩu chỉ có emoji, dấu câu gộp vào mệnh đề trước; mẩu 1 chữ sát dấu phẩy gộp sang bên kia dấu phẩy.

Notebook đọc cột `clauses` có sẵn trong file. Chỉ mục 5 chạy lại luật tách trên một mẫu, để biết mỗi mệnh đề được cắt nhờ ranh giới nào và kiểm file khớp code.

## 1. Đọc file đã xử lý

In [ ]:
df, man = doc_da_xu_ly(["comment_id", "model", "text_clean", "clauses"])
co_nd = df["text_clean"] != ""
KQ = {}   # con số chính, in ở mục 8

## 2. Số mệnh đề mỗi bình luận

**Ô code làm gì:** đếm tổng số mệnh đề trong cột `clauses`, in phân bố số mệnh đề mỗi bình luận (1, 2, 3, 4, 5+) và vẽ biểu đồ. Bình luận rỗng không có mệnh đề nào.

**Đọc kết quả:** phần lớn bình luận 1 mệnh đề là bình thường, vì bình luận YouTube thường ngắn. Nếu tỷ lệ 5+ cao thì nên đọc ví dụ ở mục 6 xem có bị cắt vụn không.

In [ ]:
so_md = df["clauses"].map(len)
n_co_nd = max(int(co_nd.sum()), 1)
tong_md = int(so_md.sum())
md_phang = pd.DataFrame([(i, m["idx"], m["text"]) for i, ds in df["clauses"].items() for m in ds],
                        columns=["dong", "idx", "text"])   # mỗi mệnh đề một dòng; "dong" là chỉ số của df
md_phang["so_chu"] = md_phang["text"].map(lambda t: len(tach_tu(t)))

KQ["Tổng số mệnh đề"] = tong_md
KQ["Trung bình mệnh đề / bình luận có nội dung"] = round(tong_md / n_co_nd, 2)
nhom = so_md[co_nd].map(lambda n: "5+" if n >= 5 else str(n))
bang_so_md = nhom.value_counts().reindex(["1", "2", "3", "4", "5+"], fill_value=0).rename("số bình luận").to_frame()
bang_so_md.index.name = "số mệnh đề"
bang_so_md["tỷ lệ (%)"] = (100 * bang_so_md["số bình luận"] / n_co_nd).round(2)
for k, dong in bang_so_md.iterrows():
    KQ[f"% bình luận có {k} mệnh đề"] = dong["tỷ lệ (%)"]
print(f"{tong_md:,} mệnh đề từ {n_co_nd:,} bình luận có nội dung (trung bình {tong_md / n_co_nd:.2f} mệnh đề / bình luận)")
display(bang_so_md)

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(bang_so_md.index, bang_so_md["số bình luận"], color="#4472c4")
ax.set_xlabel("số mệnh đề trong một bình luận")
ax.set_ylabel("số bình luận")
ax.set_title("Số mệnh đề mỗi bình luận")
plt.show()

## 3. Độ dài mệnh đề

**Ô code làm gì:** phân bố số chữ mỗi mệnh đề; % mệnh đề có từ 1 chữ trở xuống (gồm bình luận chỉ có emoji) và % mệnh đề từ `MENH_DE_DAI` chữ trở lên; các mệnh đề ngắn đó gặp nhiều nhất.

**Đọc kết quả:**
- Mệnh đề 1 chữ ("đẹp", "ok") thường không nêu khía cạnh: khi lấy mẫu gán nhãn có thể bỏ bớt.
- Mệnh đề quá dài có thể đang chứa nhiều ý mà luật chưa cắt được; xem mục 7.

In [ ]:
MENH_DE_DAI = 30   # từ bao nhiêu chữ thì coi là mệnh đề dài

KQ["Trung vị số chữ / mệnh đề"] = float(md_phang["so_chu"].median()) if len(md_phang) else None
KQ["% mệnh đề ≤ 1 chữ (gồm mệnh đề chỉ có emoji)"] = round(100 * (md_phang["so_chu"] <= 1).mean(), 2)
KQ[f"% mệnh đề ≥ {MENH_DE_DAI} chữ"] = round(100 * (md_phang["so_chu"] >= MENH_DE_DAI).mean(), 2)
display(md_phang["so_chu"].describe(percentiles=[.25, .5, .75, .9, .99]).round(1).to_frame("số chữ / mệnh đề"))
print(f"Mệnh đề ≤ 1 chữ: {(md_phang['so_chu'] <= 1).sum():,} ({100 * (md_phang['so_chu'] <= 1).mean():.2f}%) | "
      f"≥ {MENH_DE_DAI} chữ: {(md_phang['so_chu'] >= MENH_DE_DAI).sum():,} ({100 * (md_phang['so_chu'] >= MENH_DE_DAI).mean():.2f}%)")

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(md_phang["so_chu"].clip(upper=60), bins=60, color="#4472c4")
ax.set_xlabel("số chữ trong một mệnh đề (từ 60 trở lên gộp vào cột cuối)")
ax.set_ylabel("số mệnh đề")
ax.set_title("Độ dài mệnh đề")
plt.show()

print("Mệnh đề ≤ 1 chữ gặp nhiều nhất:")
display(md_phang.loc[md_phang["so_chu"] <= 1, "text"].str.lower().value_counts().head(N_SHOW).rename("số lần").to_frame())

## 4. Mệnh đề được cắt ở đâu

**Ô code làm gì:** số của toàn bộ dữ liệu, lệnh tầng 2 đếm lúc chạy (manifest): mệnh đề theo loại ranh giới đứng ngay trước nó: đầu bình luận, xuống dòng, hết câu, dấu phẩy, từ nối.

**Đọc kết quả:** "đầu bình luận" bằng đúng số bình luận có nội dung. Các loại còn lại cho biết mỗi luật cắt bao nhiêu. Một loại chiếm áp đảo thì đọc ví dụ của loại đó trước.

In [ ]:
TEN_LY_DO = {L3.DAU: "đầu bình luận", L3.XUONG_DONG: "xuống dòng", L3.HET_CAU: "hết câu", L3.DAU_PHAY: "dấu phẩy", L3.TU_NOI: "từ nối"}
md_man = man["menh_de"]
bang_ly_do = pd.Series({ten: md_man.get(f"ranh giới: {k}", 0) for k, ten in TEN_LY_DO.items()}, name="số mệnh đề").to_frame()
bang_ly_do.index.name = "loại ranh giới"
bang_ly_do["% mệnh đề"] = (100 * bang_ly_do["số mệnh đề"] / max(md_man["tong_so"], 1)).round(2)
for ten, dong in bang_ly_do.iterrows():
    KQ[f"Mệnh đề bắt đầu sau ranh giới: {ten}"] = int(dong["số mệnh đề"])
display(bang_ly_do)

## 5. Từ nối, và kiểm tra file khớp code

**Ô code làm gì:** chạy lại luật tách trên `MAU` bình luận ngẫu nhiên (đặt `None` để chạy cả bộ), so vị trí mệnh đề với cột `clauses` trong file (khác nhau nghĩa là file tạo bằng code cũ). Rồi với mỗi từ nối trong `CAC_TU_NOI`, đếm số mệnh đề **bắt đầu** bằng từ đó, chia theo:
- được cắt nhờ chính từ nối;
- được cắt nhờ ranh giới khác ngay trước (", nhưng…", "⏎nhưng…").

**Đọc kết quả:** từ nối gặp 0 lần thì có thể bỏ khỏi danh sách. Thêm từ nối mới thì làm theo hướng dẫn tầng 2 mục 5.

In [ ]:
MAU = 100_000   # None = cả bộ
mau = mau_ngau_nhien(df[co_nd], MAU)
tinh_lai = mau["text_clean"].map(lambda s: L3.tach_menh_de(s, kem_norm=False, kem_ly_do=True))
lech = sum(1 for i in mau.index
           if [(m["start"], m["end"]) for m in tinh_lai[i]] != [(m["start"], m["end"]) for m in mau.at[i, "clauses"]])
if lech:
    print(f"⚠ {lech:,} bình luận: chạy lại cho mệnh đề khác cột clauses trong file. Chạy lại lệnh tầng 2.")
else:
    print(f"Chạy lại trên {len(mau):,} bình luận cho mệnh đề giống hệt cột clauses trong file.")

md_mau = pd.DataFrame([(m["text"], m["ly_do"]) for ds in tinh_lai for m in ds], columns=["text", "ly_do"])
dau_bang_tu_noi = md_mau["text"].str.lower().str.extract(rf"^({'|'.join(L3.CAC_TU_NOI)})(?!\w)", expand=False)
bang_tu_noi = (pd.crosstab(dau_bang_tu_noi.dropna(), (md_mau.loc[dau_bang_tu_noi.notna(), "ly_do"] == L3.TU_NOI)
                           .map({True: "cắt nhờ từ nối", False: "cắt nhờ ranh giới khác"}))
                 .reindex(index=list(L3.CAC_TU_NOI), columns=["cắt nhờ từ nối", "cắt nhờ ranh giới khác"], fill_value=0)
                 .fillna(0).astype(int))
bang_tu_noi.index.name, bang_tu_noi.columns.name = "từ nối ở đầu mệnh đề", None
bang_tu_noi["tổng"] = bang_tu_noi.sum(axis=1)
KQ["Mệnh đề bắt đầu bằng từ nối (trên mẫu)"] = int(bang_tu_noi["tổng"].sum())
display(bang_tu_noi.sort_values("tổng", ascending=False))

## 6. Ví dụ bình luận có nhiều mệnh đề

**Ô code làm gì:** in `N_SHOW` bình luận ngẫu nhiên có từ 2 mệnh đề trở lên; cột "mệnh đề" nối các mệnh đề bằng ` ‖ `. Đổi `random_state` để xem bộ khác.

**Đọc kết quả:** xem có chỗ nào cắt giữa một ý (vd cắt đôi "pin trâu" thành hai mảnh), hay để sót hai ý trong một mệnh đề. Gặp lỗi lặp lại thì ghi lại cho quyết định 7 ở đặc tả tầng 2.

In [ ]:
nhieu_md = pd.Series(so_md.index[so_md >= 2])
mau_md = nhieu_md.sample(min(N_SHOW, len(nhieu_md)), random_state=0)
display(pd.DataFrame({
    "model": df.loc[mau_md, "model"].values,
    "text_clean": df.loc[mau_md, "text_clean"].str.replace("\n", " ⏎ ", regex=False).values,
    "mệnh đề": [" ‖ ".join(m["text"].replace("\n", " ⏎ ") for m in df.at[i, "clauses"]) for i in mau_md],
}))

## 7. Chỗ có thể bỏ sót

**Để làm gì:** "mà", "còn" cố ý không dùng làm ranh giới vì nhiều nghĩa ("máy còn mới"). Nhưng có lúc chúng là từ nối tương phản thật ("pin trâu còn camera thì tệ"). Mục này tìm mệnh đề dài có chứa hai chữ đó để nhóm đọc và quyết có cần thêm luật không.

**Ô code làm gì:** trong các mệnh đề từ `MENH_DE_DAI` chữ trở lên, đếm và in ví dụ mệnh đề có chữ "mà" hoặc "còn" đứng riêng.

In [ ]:
dai = md_phang[md_phang["so_chu"] >= MENH_DE_DAI]
co_ma_con = dai["text"].str.contains(r"(?<!\w)(?:mà|còn)(?!\w)", case=False, regex=True)
KQ[f"Mệnh đề ≥ {MENH_DE_DAI} chữ có 'mà'/'còn'"] = int(co_ma_con.sum())
print(f"Mệnh đề ≥ {MENH_DE_DAI} chữ: {len(dai):,}; trong đó có 'mà' hoặc 'còn': {co_ma_con.sum():,}")
if co_ma_con.any():
    vi_du_dai = dai[co_ma_con].sample(min(20, int(co_ma_con.sum())), random_state=0)
    display(vi_du_dai.assign(text=vi_du_dai["text"].str.replace("\n", " ⏎ ", regex=False).str.slice(0, 250))[["text", "so_chu"]])

## 8. Tổng hợp số liệu

**Đọc kết quả:** các con số chính của notebook này trên một bảng, để chép vào báo cáo `docs/2-tien-xu-ly/bao-cao.md` mục 3.6.

In [ ]:
display(pd.Series(KQ, name="giá trị").astype(str).to_frame())